# E1.0 · What "trustworthy AI" actually means

**Function E — Governance, Risk, Compliance & the CISO Office → The GRC Practitioner (Risk & Control)**  ·  *Security of AI*

Builds on **[D2.8 · Regulatory clock](https://spbreed.github.io/cyber-commons/lessons/D2.8.html)**.

| | |
|---|---|
| Open-source tooling | NIST AI RMF |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


"Trustworthy AI" is used as a value, and values do not have owners. Every
serious framework — NIST AI RMF chief among them — converges on roughly seven
dimensions:

| Dimension | The question it answers |
|---|---|
| **Valid and reliable** | Does it do what it claims, repeatably? |
| **Safe** | Can it cause physical, financial or psychological harm? |
| **Secure and resilient** | Can it be attacked, and does it degrade gracefully? |
| **Accountable and transparent** | Can you say who is responsible, and show your working? |
| **Explainable and interpretable** | Can you say why it produced this output? |
| **Privacy-enhanced** | Whose data is in it, on what basis, for how long? |
| **Fair, with harmful bias managed** | Does it distribute error evenly across people? |

Learning the list is the easy half and takes an afternoon.

The hard half is that **each dimension needs a named owning function**, and in
most organisations two or three of them have either no owner or three. "Everyone
owns trustworthy AI" is operationally identical to nobody owning it, and it fails
in a predictable direction: the dimensions with obvious homes get controls, and
the ones that sit between functions get a policy sentence.

Below, the seven dimensions are assigned across five functions, and then the
assignment is checked — because an ownership map with gaps is more useful than
one without, provided you can see the gaps.

## 2 · The seven dimensions, and who could own each

In [ ]:
DIMENSIONS = {
 "valid_and_reliable":     "does it do what it claims, repeatably",
 "safe":                   "can it cause physical, financial or psychological harm",
 "secure_and_resilient":   "can it be attacked, does it degrade gracefully",
 "accountable_transparent":"who is responsible, and can you show your working",
 "explainable":            "why did it produce this output",
 "privacy_enhanced":       "whose data, on what basis, for how long",
 "fair_bias_managed":      "is error distributed evenly across people",
}
FUNCTIONS = ["legal", "compliance", "privacy", "cyber", "model_risk"]

print(f"{'dimension':26s}the question it answers")
for d in sorted(DIMENSIONS):
    print(f"{d:26s}{DIMENSIONS[d]}")
print(f"\nfunctions available to own them: {FUNCTIONS}")

## 3 · Assign, then look for the gaps\n\nA typical assignment in a large organisation. Not a recommended one — an observed one.

In [ ]:
OWNERS = {
 "valid_and_reliable":      ["model_risk"],
 "safe":                    [],                       # nobody
 "secure_and_resilient":    ["cyber"],
 "accountable_transparent": ["compliance", "legal", "model_risk"],   # three
 "explainable":             ["model_risk"],
 "privacy_enhanced":        ["privacy"],
 "fair_bias_managed":       [],                       # nobody
}
print(f"{'dimension':26s}{'owners':34s}status")
for d in sorted(DIMENSIONS):
    o = OWNERS[d]
    status = ("UNOWNED" if not o else
              "contested" if len(o) > 1 else "clear")
    print(f"{d:26s}{', '.join(o) or '-':34s}{status}")

unowned = sorted(d for d in DIMENSIONS if not OWNERS[d])
contested = sorted(d for d in DIMENSIONS if len(OWNERS[d]) > 1)
print(f"\nunowned   : {unowned}")
print(f"contested : {contested}")
assert unowned and contested

## 4 · Where it breaks — the two failure modes look different and are not\n\nAn unowned dimension produces no control. A contested one produces three partial controls and no complete one.

In [ ]:
def controls_for(dimension, owners):
    """Each owner builds the part of the control they can see from their seat."""
    coverage = {"legal": 0.3, "compliance": 0.35, "privacy": 0.4,
                "cyber": 0.9, "model_risk": 0.8}
    if not owners:
        return 0.0, "no control exists"
    if len(owners) == 1:
        return coverage[owners[0]], f"{owners[0]} builds it end to end"
    best = max(coverage[o] for o in owners)
    return best, f"{len(owners)} partial controls, none complete, best is {best:.0%}"

print(f"{'dimension':26s}{'coverage':>10s}  what actually got built")
for d in sorted(DIMENSIONS):
    cov, why = controls_for(d, OWNERS[d])
    print(f"{d:26s}{cov:>9.0%}  {why}")
print()
weakest = sorted((controls_for(d, OWNERS[d])[0], d) for d in DIMENSIONS)[:3]
print("weakest three:", [d for _, d in weakest])
print()
print("The unowned dimensions are at zero, which at least is visible. The")
print("contested one is worse: three functions each report that it is covered,")
print("and each is telling the truth about their part.")

## 5 · The control — one accountable owner, others named as contributors

In [ ]:
FIXED = {
 "valid_and_reliable":      ("model_risk", ["cyber"]),
 "safe":                    ("business_owner", ["legal", "compliance"]),
 "secure_and_resilient":    ("cyber", ["model_risk"]),
 "accountable_transparent": ("compliance", ["legal", "model_risk"]),
 "explainable":             ("model_risk", ["compliance"]),
 "privacy_enhanced":        ("privacy", ["cyber", "legal"]),
 "fair_bias_managed":       ("model_risk", ["compliance", "legal"]),
}
print(f"{'dimension':26s}{'accountable':16s}contributors")
for d in sorted(FIXED):
    owner, contrib = FIXED[d]
    print(f"{d:26s}{owner:16s}{', '.join(contrib)}")

still_unowned = [d for d in DIMENSIONS if not FIXED[d][0]]
print(f"\ndimensions with no accountable owner: {still_unowned or 'none'}")
print()
print("Note the seat that had to be invented: `business_owner`. Safety of a use")
print("case is not a control function's decision - it belongs to whoever chose")
print("to deploy it, and if that seat is empty the other five are governing an")
print("orphan.")
assert not still_unowned

## 6 · Verify — every dimension resolves to a person who can be asked

In [ ]:
def audit_question(dimension):
    owner, contrib = FIXED[dimension]
    return {"dimension": dimension,
            "who_do_i_ask": owner,
            "who_else_must_agree": contrib,
            "answerable": bool(owner)}

for d in sorted(DIMENSIONS):
    q = audit_question(d)
    print(f"   {d:26s}ask {q['who_do_i_ask']:16s}"
          f"agreed with {', '.join(q['who_else_must_agree']) or '-'}")
print()
print(f"answerable for all seven: {all(audit_question(d)['answerable'] for d in DIMENSIONS)}")
print()
print("That is the whole test. An auditor asks one question per dimension, and")
print("every question resolves to a person. E1.10 takes the same five functions")
print("and maps the controls each one actually operates.")
assert all(audit_question(d)["answerable"] for d in DIMENSIONS)

## What you just proved

The seven trustworthy-AI dimensions print with the question each answers. A typical assignment leaves two unowned and one contested between three functions — and the contested one is shown to be worse than the unowned ones, because three functions each honestly report their part as covered. The fix names one accountable owner per dimension and has to invent the business-owner seat to do it.

## Your turn

Write your organisation's names against the seven dimensions. The interesting rows are the blanks and the ones where you wrote three names — and the second kind is the one that will surprise you in an audit.

---

**Next → [E1.1 · Why point-in-time control testing fails for AI](https://spbreed.github.io/cyber-commons/lessons/E1.1.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E1.0.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E1.0.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*